# Approximating LeNet logits with affine and quadratic maps
Andrey Melnikov · MNIST experiments

This runnable demo fits the **cached training logits** supplied with the project. No teacher weights are needed. A deterministic held-out portion of the original training split is used for evaluation; this is not the official MNIST test result reported in the original notebooks.

The quadratic model uses $z=V(x/255-\mu)$ and $\hat Y=[z,\{z_i z_j\}_{i\le j},1]W$. PCA and the readout are fitted only on the fitting subset. Start with 2,000 fitting samples and 20 PCs; the original experiment used 60,000 samples and 100 PCs.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, shutil
# Colab downloads the public repository automatically. Locally, use a checkout.
ROOT = Path.cwd()
if not (ROOT / "approximation.py").exists():
    ROOT = Path.cwd() / "lenet-examples"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/andreymathcom/lenet-examples.git", str(ROOT)], check=True)
if not (ROOT / "data/Y_lenet_mnist.npy").exists():
    with zipfile.ZipFile(ROOT / "lenet-data.zip") as archive:
        for member in archive.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError("Unsafe archive path")
        archive.extractall(ROOT)
(ROOT / "models").mkdir(exist_ok=True)
if not (ROOT / "models/lenet.onnx").exists():
    shutil.copyfile(ROOT / "lenet.onnx", ROOT / "models/lenet.onnx")
sys.path.insert(0, str(ROOT))
print("Project:", ROOT)


In [ ]:
import numpy as np
from approximation import load_data, fit_affine, predict_affine, fit_quadratic, predict_quadratic
X, X_test, labels, labels_test, Y = load_data(ROOT)
rng = np.random.default_rng(42)
order = rng.permutation(len(X))
N_FIT, N_VALID, PCA_K = 2000, 1000, 20
fit, valid = order[:N_FIT], order[N_FIT:N_FIT+N_VALID]
print('Fit:', len(fit), 'Validation:', len(valid), 'PCA rank:', PCA_K)


In [ ]:
affine = fit_affine(X[fit], Y[fit])
quadratic = fit_quadratic(X[fit], Y[fit], PCA_K)
print('Held-out training-subset results (not official test-set results)')
for name, pred in [('Cached teacher', Y[valid]),
                   ('Affine', predict_affine(X[valid], affine)),
                   ('Quadratic', predict_quadratic(X[valid], quadratic))]:
    accuracy = 100*np.mean(pred.argmax(1) == labels[valid])
    agreement = 100*np.mean(pred.argmax(1) == Y[valid].argmax(1))
    rmse = np.sqrt(np.mean((pred-Y[valid])**2))
    print(f'{name:15s} accuracy={accuracy:.3f}% teacher agreement={agreement:.3f}% logit RMSE={rmse:.4f}')


In [ ]:
mu, V, W = quadratic
np.savez(ROOT/'models/quadratic_demo.npz', mu=mu, V=V, W=W)
print('Saved fitted demonstration model; these are not the original full-data weights.')
